# Fake News Detector
### Approach:
* TF-IDF with ngrams
* Logistic Regression 

In [1]:
# Basic imports
import pandas as pd
# Pre-processing
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
# Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/aparajaya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Data Loading 
* Load and examine dataset
* Convert datatypes
* Treat missing values

In [2]:
df = pd.read_csv("Dataset.csv")

In [3]:
# Examining Dataset 
print("Dataset shape:",df.shape)
print("\nSample: ")
df.head(1)

Dataset shape: (72134, 4)

Sample: 


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1


In [4]:
# Checking and converting datatypes
print("Original Datatypes:\n",df.dtypes)
df['title'] = df['title'].astype('string')
df['text'] = df['text'].astype('string')
print("\nChanged Datatypes:\n",df.dtypes)

Original Datatypes:
 Unnamed: 0     int64
title         object
text          object
label          int64
dtype: object

Changed Datatypes:
 Unnamed: 0             int64
title         string[python]
text          string[python]
label                  int64
dtype: object


In [5]:
# Treating missing values
print("Number of missing values per column:\n",df.isna().sum())
df['title'] = df['title'].fillna('')
df['text'] = df['text'].fillna('')
print("\nAfter treating missing values-")
print("Number of missing values per column:\n",df.isna().sum())

Number of missing values per column:
 Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

After treating missing values-
Number of missing values per column:
 Unnamed: 0    0
title         0
text          0
label         0
dtype: int64


## Preprocessing

In [6]:
# Convert to Lowercase

df['text'] = df['text'].str.lower()
df['title'] = df['title'].str.lower()

print("After converting to lowercase:\n")
df.head(2)

After converting to lowercase:



,Unnamed: 0,title,text,label
0,0,law enforcement on high alert following threat...,no comment is expected from barack obama membe...,1
1,1,,did they post their votes for hillary already?,1


In [7]:
# Remove punctuation

df['title'] = df['title'].str.replace(r'[^\w\s]+', '', regex=True) 
df['text'] = df['text'].str.replace(r'[^\w\s]+', '', regex=True) 

print("After removing punctuation:\n")
df.head(2)

After removing punctuation:



,Unnamed: 0,title,text,label
0,0,law enforcement on high alert following threat...,no comment is expected from barack obama membe...,1
1,1,,did they post their votes for hillary already,1


In [8]:
# Remove stopwords (Conditional)

toggle = True
if toggle:
    stop_words = set(stopwords.words('english'))

    df['text'] = df['text'].apply(
        lambda x: " ".join([word for word in x.split() if word not in stop_words])
    )
    df['title'] = df['title'].apply(
        lambda x: " ".join([word for word in x.split() if word not in stop_words])
    )

    print("After removing stopwords\n")
df.head(2)

After removing stopwords



,Unnamed: 0,title,text,label
0,0,law enforcement high alert following threats c...,comment expected barack obama members fyf911 f...,1
1,1,,post votes hillary already,1


In [10]:
# Check if pre-processing caused too many empty rows
print("No of empty title rows: ",(df['title'].str.len() == 0).sum())
print("No of empty text rows: ",(df['text'].str.len() == 0).sum())
count = ((df['title'].str.len() == 0) & (df['text'].str.len() == 0)).sum()
print("No of rows with empty title AND text: ",count)

No of empty title rows:  563
No of empty text rows:  786
No of rows with empty title AND text:  1


## Vectorization 

In [16]:
# TF-IDF with ngrams

# Initialize the TF-IDF Vectorizer
vectorizer = TfidfVectorizer(min_df=5, ngram_range=(1,2), max_features=500000)

# Fit and transform the data
tfidf_title = vectorizer.fit_transform(df['title'])
tfidf_text = vectorizer.fit_transform(df['text'])

# Matrix inspection
# Shape = (number_of_documents, number_of_features)
print(tfidf_title.shape)
print(tfidf_text.shape)

(72134, 23007)
(72134, 500000)


## Model
### Logistic Regression